In [6]:
"""
01_generate_training_data.py
=============================
Generates 11 years (2015-01-01 → 2026-01-01) of realistic hourly RAN
telemetry that exactly mirrors the schema of ran_dataset_site001.parquet.

Key design decisions
--------------------
• Hourly rows per site  →  96,360 rows per site
• 3 cells per site, wide format  (cell_1_*, cell_2_*, cell_3_*)
• Failure rows = exactly 25 % of the dataset
• Technology evolution: 2G(2015) → 3G(2016) → 4G(2017-2021) → 5G(2022+)
• Diurnal, weekly, seasonal and long-term traffic growth patterns
• 10 distinct failure scenarios, each with realistic pre-failure degradation,
  active failure and recovery phases
• All numeric ranges match values seen in the real parquet file

Run in Google Colab
-------------------
    !pip install -q pyarrow pandas numpy
    # then run this file
"""

# ── 0. Install ─────────────────────────────────────────────────────────────────
import subprocess, sys
def _pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
_pip("pyarrow","pandas","numpy")

# ── 1. Imports ─────────────────────────────────────────────────────────────────
import os
import uuid, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
START       = datetime(2015, 1, 1, tzinfo=timezone.utc)
END         = datetime(2026, 1, 1, tzinfo=timezone.utc)
FREQ        = "h"                       # hourly
TARGET_FAIL = 0.25                      # 25 % failure rows
N_CELLS     = 3
SEED        = 2015
OUTPUT      = "ran_training_data.parquet"

rng = np.random.default_rng(SEED)

# ── Site metadata ──────────────────────────────────────────────────────────────
SITES = [
    dict(site_id="SITE_001", site_name="CAIRO_TOWER_01",
         lat=30.0444, lon=31.2357, region="Cairo",    vendor="Ericsson"),
    dict(site_id="SITE_002", site_name="ALEX_TOWER_01",
         lat=31.2001, lon=29.9187, region="Alexandria", vendor="Huawei"),
    dict(site_id="SITE_003", site_name="GZA_TOWER_01",
         lat=30.0131, lon=31.2089, region="Giza",     vendor="Nokia"),
]

# Cell technology per cell per site (evolves over years)
CELL_TECH = {
    "SITE_001": ["4G", "4G", "5G"],
    "SITE_002": ["4G", "4G", "4G"],
    "SITE_003": ["4G", "5G", "5G"],
}
CELL_BW = {
    "SITE_001": [20, 20, 100],
    "SITE_002": [20, 20, 20],
    "SITE_003": [20, 100, 100],
}

# ── Failure scenario catalogue ─────────────────────────────────────────────────
# (name, duration_hours, affected_cells, severity 0-1)
SCENARIOS = [
    ("BBU_CPU_SATURATION",        24, [1,2,3], 0.75),
    ("BBU_FULL_OUTAGE",           12, [1,2,3], 1.00),
    ("CELL1_HIGH_BLER",           36, [1],     0.60),
    ("CELL2_HANDOVER_STORM",      18, [2],     0.65),
    ("GENERATOR_FUEL_DEPLETION",  48, [1,2,3], 0.80),
    ("FULL_SITE_POWER_OUTAGE",     8, [1,2,3], 1.00),
    ("MICROWAVE_BACKHAUL_FADE",   30, [3],     0.55),
    ("BACKHAUL_FIBER_CUT",        16, [1,2],   0.85),
    ("ANT1_PHYSICAL_TILT_DRIFT",  72, [1],     0.40),
    ("ANT2_VSWR_ALARM",           48, [2],     0.50),
]

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def clip(arr, lo, hi): return np.clip(arr, lo, hi)

def sinusoidal(n, amplitude, period, phase=0):
    t = np.arange(n)
    return amplitude * np.sin(2*np.pi*(t+phase)/period)

def growth_factor(timestamps):
    """Long-term traffic growth: ~80 % total over 11 years."""
    year_frac = (timestamps - pd.Timestamp("2015-01-01", tz="UTC")) \
                / pd.Timedelta(days=365)
    return 1.0 + 0.8 * np.clip(np.array(year_frac) / 11, 0, 1)

def diurnal(timestamps):
    """Peak at 20:00, trough at 04:00."""
    h = timestamps.hour
    return 0.5 + 0.5 * np.sin(2*np.pi*(h - 4)/24)

def technology_for_year(year, cell_idx, site_id):
    """Return technology string adjusted for deployment era."""
    base = CELL_TECH[site_id][cell_idx - 1]
    if year < 2017:
        return "2G" if cell_idx == 3 else "3G"
    if year < 2022 and base == "5G":
        return "4G"
    return base

# ══════════════════════════════════════════════════════════════════════════════
# FAILURE SCHEDULE
# Deterministically assigns ~25 % of hours to a failure scenario.
# ══════════════════════════════════════════════════════════════════════════════

def build_failure_schedule(timestamps, site_id):
    """
    Returns an array of scenario labels (str) aligned with timestamps.
    Ensures exactly TARGET_FAIL fraction are non-NORMAL.
    """
    n = len(timestamps)
    labels = np.full(n, "NORMAL", dtype=object)
    target_fail_count = int(n * TARGET_FAIL)

    fail_assigned = 0
    attempt = 0
    while fail_assigned < target_fail_count and attempt < 50000:
        attempt += 1
        scenario, duration, _, _ = SCENARIOS[rng.integers(len(SCENARIOS))]
        # Random start, leave room for recovery window (10 % extra)
        start_i = int(rng.integers(0, max(1, n - int(duration * 1.1))))
        end_i   = min(n, start_i + duration)
        # Only overwrite NORMAL hours
        mask = (labels[start_i:end_i] == "NORMAL")
        new_fail = mask.sum()
        if fail_assigned + new_fail > target_fail_count * 1.05:
            continue
        labels[start_i:end_i][mask] = scenario
        fail_assigned += new_fail

    # Trim to exact 25 %: convert excess failures back to NORMAL
    fail_idx = np.where(labels != "NORMAL")[0]
    excess   = len(fail_idx) - target_fail_count
    if excess > 0:
        to_normal = rng.choice(fail_idx, size=excess, replace=False)
        labels[to_normal] = "NORMAL"

    actual_pct = 100 * (labels != "NORMAL").mean()
    print(f"    {site_id}: {(labels!='NORMAL').sum():,} failure rows "
          f"({actual_pct:.1f}%) / {n:,} total")
    return labels

# ══════════════════════════════════════════════════════════════════════════════
# CELL KPI GENERATOR
# ══════════════════════════════════════════════════════════════════════════════

def cell_kpis(n, timestamps, cell_idx, site_id, scenario_labels):
    """Generate all cell_N_* KPI columns for one cell."""
    tech   = [technology_for_year(ts.year, cell_idx, site_id)
              for ts in timestamps]
    bw     = CELL_BW[site_id][cell_idx - 1]
    grow   = growth_factor(timestamps)
    diurn  = np.array([diurnal(ts) for ts in timestamps])
    weekly = 0.9 + 0.1 * np.array([(1 if ts.weekday() < 5 else 0.7)
                                    for ts in timestamps])
    season = 1.0 + 0.08 * np.sin(2*np.pi*np.arange(n)/8760)  # yearly cycle

    base_load = grow * diurn * weekly * season  # 0..~1.8

    # ── Base KPIs ──────────────────────────────────────────────────────────────
    prb   = clip(35 + 40*base_load + rng.normal(0,3,n), 5, 97)
    tpdl  = clip(60 + 120*base_load*grow + rng.normal(0,8,n), 0, 600)
    tpul  = clip(15 + 35*base_load*grow  + rng.normal(0,4,n), 0, 150)
    se    = clip(2.5 + 2.5*base_load + rng.normal(0,0.3,n), 0.5, 7.5)
    sinr  = clip(14 - 4*base_load + rng.normal(0,2,n), -5, 28)
    rsrp  = clip(-88 - 4*base_load + rng.normal(0,3,n), -120, -65)
    rsrq  = clip(-10 - 2*base_load + rng.normal(0,1.5,n), -15, -3)
    cqi   = clip(9 - 2*base_load + rng.normal(0,1,n), 2, 15)
    blerD = clip(0.8 + 1.5*base_load + np.abs(rng.normal(0,0.3,n)), 0.1, 10)
    blerU = clip(0.8 + 1.5*base_load + np.abs(rng.normal(0,0.3,n)), 0.1, 10)
    harq  = clip(1.5 + 2*blerD/10   + np.abs(rng.normal(0,0.3,n)), 0.5, 20)
    latD  = clip(18 + 8*base_load   + rng.normal(0,2,n), 5, 45)
    latU  = clip(16 + 6*base_load   + rng.normal(0,2,n), 5, 45)
    hoAtt = clip(40 + 80*base_load  + rng.normal(0,8,n), 0, 700).astype(int)
    hoSR  = clip(96 + rng.normal(0,1,n), 80, 99.5)
    hoF   = np.maximum(0, (hoAtt * (1 - hoSR/100)).astype(int))
    rrcA  = clip(400+600*base_load  + rng.normal(0,40,n), 0, 5000).astype(int)
    rrcSR = clip(97.5 + rng.normal(0,0.5,n), 85, 99.5)
    erab  = clip(97.5 + rng.normal(0,0.5,n), 85, 99.5)
    cdr   = clip(0.3 + 0.8*base_load + np.abs(rng.normal(0,0.15,n)), 0.05, 5)
    abr   = clip(0.3 + 0.8*base_load + np.abs(rng.normal(0,0.15,n)), 0.05, 5)
    users = clip(80 + 200*base_load  + rng.normal(0,15,n), 0, 600).astype(int)
    conn  = clip(users * 1.2 + rng.normal(0,5,n), 0, 650).astype(int)
    status = np.full(n, "UP", dtype=object)
    opst   = np.full(n, "ACTIVE", dtype=object)

    # ── Failure injection ──────────────────────────────────────────────────────
    for i, sc in enumerate(scenario_labels):
        if sc == "NORMAL":
            continue
        # Find this scenario's spec
        spec = next((s for s in SCENARIOS if s[0] == sc), None)
        if spec is None:
            continue
        _, dur, affected_cells, sev = spec

        if cell_idx not in affected_cells:
            continue

        # Pre-degradation window (25 % of duration before this hour)
        # We apply degradation based on position within consecutive block
        # Approximate: treat each failure hour independently with full severity
        ns = sev  # numeric severity 0-1

        # SINR / RSRP degradation
        sinr[i]  = clip(sinr[i]  - 15*ns + rng.normal(0,1), -5, 28)
        rsrp[i]  = clip(rsrp[i]  - 12*ns + rng.normal(0,2), -120, -65)
        rsrq[i]  = clip(rsrq[i]  - 4*ns  + rng.normal(0,0.5), -15, -3)
        cqi[i]   = clip(cqi[i]   - 5*ns  + rng.normal(0,0.5), 2, 15)

        # Error rate inflation
        blerD[i] = clip(blerD[i] + 12*ns + rng.normal(0,0.5), 0.1, 10)
        blerU[i] = clip(blerU[i] + 12*ns + rng.normal(0,0.5), 0.1, 10)
        harq[i]  = clip(harq[i]  + 8*ns  + rng.normal(0,0.3), 0.5, 20)
        cdr[i]   = clip(cdr[i]   + 3*ns  + rng.normal(0,0.2), 0.05, 5)
        abr[i]   = clip(abr[i]   + 3*ns  + rng.normal(0,0.2), 0.05, 5)

        # Latency spike
        latD[i]  = clip(latD[i]  + 20*ns + rng.normal(0,2), 5, 45)
        latU[i]  = clip(latU[i]  + 18*ns + rng.normal(0,2), 5, 45)

        # Throughput drop
        tpdl[i]  = clip(tpdl[i]  * (1 - 0.7*ns), 0, 600)
        tpul[i]  = clip(tpul[i]  * (1 - 0.7*ns), 0, 150)
        prb[i]   = clip(prb[i]   * (1 + 0.3*ns), 5, 97)  # PRB spikes under failure

        # Status for full outages
        if ns >= 0.95:
            status[i] = "DOWN"
            opst[i]   = "INACTIVE"
            users[i]  = 0; conn[i] = 0
        elif ns >= 0.6:
            status[i] = "DEGRADED"

    prefix = f"cell_{cell_idx}_"
    return {
        f"{prefix}status":                           status,
        f"{prefix}op_state":                         opst,
        f"{prefix}active_users":                     users,
        f"{prefix}connected_users":                  conn,
        f"{prefix}prb_utilization_percent":          np.round(prb,  2),
        f"{prefix}throughput_downlink_mbps":         np.round(tpdl, 2),
        f"{prefix}throughput_uplink_mbps":           np.round(tpul, 2),
        f"{prefix}spectral_efficiency_bps_per_hz":   np.round(se,   3),
        f"{prefix}rsrp_dbm":                         np.round(rsrp, 2),
        f"{prefix}rsrq_db":                          np.round(rsrq, 2),
        f"{prefix}sinr_db":                          np.round(sinr, 2),
        f"{prefix}cqi_avg":                          np.round(cqi,  2),
        f"{prefix}bler_downlink_percent":            np.round(blerD,3),
        f"{prefix}bler_uplink_percent":              np.round(blerU,3),
        f"{prefix}harq_retransmission_rate_percent": np.round(harq, 3),
        f"{prefix}latency_downlink_ms":              np.round(latD, 2),
        f"{prefix}latency_uplink_ms":                np.round(latU, 2),
        f"{prefix}handover_attempts":                hoAtt,
        f"{prefix}handover_success_rate_percent":    np.round(hoSR, 2),
        f"{prefix}handover_failures":                hoF,
        f"{prefix}rrc_connection_attempts":          rrcA,
        f"{prefix}rrc_success_rate_percent":         np.round(rrcSR,2),
        f"{prefix}erab_setup_success_rate_percent":  np.round(erab, 2),
        f"{prefix}call_drop_rate_percent":           np.round(cdr,  3),
        f"{prefix}abnormal_release_rate_percent":    np.round(abr,  3),
        f"{prefix}technology":                       tech,
        f"{prefix}bandwidth_mhz":                   bw,
    }

# ══════════════════════════════════════════════════════════════════════════════
# SITE-LEVEL COLUMN GENERATORS
# ══════════════════════════════════════════════════════════════════════════════

def bbu_cols(n, timestamps, scenario_labels):
    grow  = growth_factor(timestamps)
    diurn = np.array([diurnal(ts) for ts in timestamps])
    cpu   = clip(30 + 40*diurn*grow + rng.normal(0,5,n), 5, 99)
    mem   = clip(25 + 35*diurn*grow + rng.normal(0,4,n), 5, 97)
    disk  = clip(20 + 0.0005*np.arange(n) + rng.normal(0,2,n), 5, 92)
    plat  = clip(8  + 5*diurn + rng.normal(0,2,n), 3, 55)
    ausers= clip(200+600*diurn*grow + rng.normal(0,30,n), 0, 1800).astype(int)
    cplat = clip(6  + 4*diurn + rng.normal(0,1.5,n), 3, 35)
    uplat = clip(7  + 4*diurn + rng.normal(0,1.5,n), 3, 45)

    bbu_st = np.full(n, "UP", dtype=object)
    bbu_op = np.full(n, "ACTIVE", dtype=object)

    for i, sc in enumerate(scenario_labels):
        if "BBU" in sc:
            if "OUTAGE" in sc:
                cpu[i]=99; mem[i]=99; bbu_st[i]="DOWN"; bbu_op[i]="INACTIVE"
            else:  # CPU_SATURATION
                cpu[i]=clip(cpu[i]+35+rng.normal(0,3), 5, 99)
                mem[i]=clip(mem[i]+25+rng.normal(0,3), 5, 97)
                bbu_st[i]="DEGRADED"

    return {
        "bbu_status": bbu_st, "bbu_op_state": bbu_op,
        "bbu_cpu_utilization_percent":    np.round(cpu,  2),
        "bbu_memory_utilization_percent": np.round(mem,  2),
        "bbu_disk_usage_percent":         np.round(disk, 2),
        "bbu_process_latency_ms":         np.round(plat, 2),
        "bbu_active_users":               ausers,
        "bbu_control_plane_latency_ms":   np.round(cplat,2),
        "bbu_user_plane_latency_ms":      np.round(uplat,2),
    }


def ru_ant_cols(n, scenario_labels):
    """RU and ANT columns — realistic but not used by the cell-only model."""
    d = {}
    for ru in range(1, 4):
        st = np.full(n, "UP", dtype=object)
        for i, sc in enumerate(scenario_labels):
            if "OUTAGE" in sc or "BBU_FULL" in sc:
                st[i] = "DOWN"
        d.update({
            f"ru_{ru}_status":                st,
            f"ru_{ru}_op_state":              np.where(st=="DOWN","INACTIVE","ACTIVE"),
            f"ru_{ru}_temperature_c":         np.round(clip(35+sinusoidal(n,8,8760)+rng.normal(0,3,n),20,65),2),
            f"ru_{ru}_tx_power_watts":        np.round(clip(18+rng.normal(0,2,n),5,40),2),
            f"ru_{ru}_rx_signal_strength_dbm":np.round(clip(-65+rng.normal(0,5,n),-100,-30),2),
            f"ru_{ru}_vswr":                  np.round(clip(1.2+np.abs(rng.normal(0,0.1,n)),1.0,3.0),3),
            f"ru_{ru}_current_ampere":        np.round(clip(5+rng.normal(0,1,n),1,20),2),
            f"ru_{ru}_voltage_volt":          np.round(clip(48+rng.normal(0,0.5,n),44,54),2),
            f"ru_{ru}_packet_error_rate":     np.round(np.abs(rng.normal(0,0.001,n)),5),
            f"ru_{ru}_throughput_mbps":       np.round(clip(100+rng.normal(0,30,n),10,500),2),
        })
    for ant in range(1, 4):
        st = np.full(n, "UP", dtype=object)
        op = np.full(n, "ACTIVE", dtype=object)
        vswr_raise = np.zeros(n)
        for i, sc in enumerate(scenario_labels):
            if f"ANT{ant}" in sc and "VSWR" in sc:
                vswr_raise[i] = 1.5
            if f"ANT{ant}" in sc and "TILT" in sc:
                pass  # no status change, just KPI drift
            if "OUTAGE" in sc:
                st[i] = "DOWN"; op[i] = "INACTIVE"
        d.update({
            f"ant_{ant}_tilt_degree":   np.round(clip(4+rng.normal(0,0.5,n)+
                                         (np.arange(n)*0.0002 if ant==1 else 0),0,15),2),
            f"ant_{ant}_azimuth_degree":float([120,240,0][ant-1]),
            f"ant_{ant}_mimo_layers":   rng.choice([2,4,8],n),
            f"ant_{ant}_status":        st,
            f"ant_{ant}_op_state":      op,
            f"ant_{ant}_rssi_dbm":      np.round(clip(-70+rng.normal(0,5,n),-100,-35),2),
            f"ant_{ant}_snr_db":        np.round(clip(14+rng.normal(0,3,n),5,35),2),
        })
    return d


def bh_cols(n, scenario_labels):
    d = {}
    for bh in range(1, 3):
        bh_type = "FIBER" if bh == 1 else "MICROWAVE"
        st = np.full(n, "UP", dtype=object)
        op = np.full(n, "ACTIVE", dtype=object)
        lat  = clip(4+rng.normal(0,1,n), 1, 20)
        loss = clip(np.abs(rng.normal(0,0.01,n)), 0, 0.1)
        util = clip(35+30*np.array([diurnal(ts) for ts in
                    pd.date_range("2015-01-01",periods=n,freq="h")])+rng.normal(0,5,n),5,90)

        for i, sc in enumerate(scenario_labels):
            if bh==1 and "FIBER" in sc:
                st[i]="DOWN"; op[i]="INACTIVE"
                lat[i]=clip(lat[i]+50,1,20); loss[i]=clip(loss[i]+0.08,0,0.1)
            if bh==2 and "MICROWAVE" in sc:
                st[i]="DEGRADED"
                lat[i]=clip(lat[i]+15,1,20); loss[i]=clip(loss[i]+0.05,0,0.1)
            if "POWER_OUTAGE" in sc or "BBU_FULL" in sc:
                st[i]="DOWN"; op[i]="INACTIVE"

        d.update({
            f"bh_{bh}_status":               st,
            f"bh_{bh}_op_state":             op,
            f"bh_{bh}_latency_ms":           np.round(lat, 3),
            f"bh_{bh}_jitter_ms":            np.round(clip(np.abs(rng.normal(1,0.3,n)),0.1,8),3),
            f"bh_{bh}_packet_loss_percent":  np.round(loss, 4),
            f"bh_{bh}_throughput_mbps":      np.round(clip(200+rng.normal(0,40,n),50,1000),2),
            f"bh_{bh}_utilization_percent":  np.round(util, 2),
            f"bh_{bh}_type":                 bh_type,
        })
    return d


def power_cols(n, timestamps, scenario_labels):
    fuel = clip(95 - np.arange(n)*0.003 + sinusoidal(n,3,8760) + rng.normal(0,1,n), 65, 100)
    bat1 = clip(95 + rng.normal(0,2,n), 80, 100)
    bat2 = clip(94 + rng.normal(0,2,n), 80, 100)
    r1v  = clip(48 + rng.normal(0,0.3,n), 46, 50)
    r1a  = clip(20 + rng.normal(0,3,n), 8, 50)
    r2v  = clip(48 + rng.normal(0,0.3,n), 46, 50)
    r2a  = clip(20 + rng.normal(0,3,n), 8, 50)
    b1t  = clip(28 + sinusoidal(n,5,8760) + rng.normal(0,2,n), 15, 48)
    b2t  = clip(27 + sinusoidal(n,5,8760) + rng.normal(0,2,n), 15, 48)

    pw_st  = np.full(n,"UP",dtype=object)
    gen_st = np.full(n,"UP",dtype=object)
    r1_st  = np.full(n,"UP",dtype=object)
    r2_st  = np.full(n,"UP",dtype=object)
    b1_st  = np.full(n,"UP",dtype=object)
    b2_st  = np.full(n,"UP",dtype=object)

    for i, sc in enumerate(scenario_labels):
        if "FUEL" in sc:
            fuel[i] = clip(fuel[i]-30+rng.normal(0,3), 65, 100)
            gen_st[i] = "DEGRADED"
        if "POWER_OUTAGE" in sc:
            pw_st[i]="DOWN"; gen_st[i]="DOWN"
            r1_st[i]="DOWN"; r2_st[i]="DOWN"
            fuel[i]=clip(fuel[i]-15, 65, 100)

    return {
        "power_status": pw_st, "gen_status": gen_st,
        "gen_fuel_pct":   np.round(fuel,2),
        "rec_1_status":   r1_st,
        "rec_1_voltage_v":np.round(r1v,2),
        "rec_1_current_a":np.round(r1a,2),
        "rec_2_status":   r2_st,
        "rec_2_voltage_v":np.round(r2v,2),
        "rec_2_current_a":np.round(r2a,2),
        "bat_1_status":   b1_st,
        "bat_1_charge_pct":np.round(bat1,2),
        "bat_1_temp_c":   np.round(b1t,2),
        "bat_2_status":   b2_st,
        "bat_2_charge_pct":np.round(bat2,2),
        "bat_2_temp_c":   np.round(b2t,2),
    }


def env_cols(n, timestamps, scenario_labels):
    # Outdoor-influenced shelter temperature
    season  = sinusoidal(n, 10, 8760)
    diurn_t = sinusoidal(n, 5, 24)
    t1 = clip(28 + season + diurn_t + rng.normal(0,2,n), 18, 55)
    t2 = clip(27 + season + diurn_t + rng.normal(0,2,n), 18, 55)
    hum = clip(55 + sinusoidal(n,20,8760) + rng.normal(0,5,n), 20, 95)
    door = rng.integers(0, 2, n)
    smoke_v = np.zeros(n, int)
    est = np.full(n,"UP",dtype=object)
    for i, sc in enumerate(scenario_labels):
        if "OUTAGE" in sc:
            t1[i]=clip(t1[i]+8, 18, 55); t2[i]=clip(t2[i]+8, 18, 55)
    return {
        "env_status":   est,
        "env_temp_1_c": np.round(t1, 2),
        "env_temp_2_c": np.round(t2, 2),
        "env_humidity": np.round(hum, 2),
        "door_open":    door,
        "smoke":        smoke_v,
    }


# ══════════════════════════════════════════════════════════════════════════════
# SITE GENERATOR
# ══════════════════════════════════════════════════════════════════════════════

def generate_site(site_meta, timestamps):
    n  = len(timestamps)
    sid = site_meta["site_id"]
    print(f"  Generating {sid} ({n:,} rows) …")

    labels = build_failure_schedule(timestamps, sid)
    ts_arr = pd.DatetimeIndex(timestamps)

    rows = {}

    # Meta
    rows["message_id"]     = [str(uuid.uuid4()) for _ in range(n)]
    rows["timestamp"]      = timestamps
    rows["sequence_number"]= np.arange(1, n+1)
    rows["scenario_label"] = labels
    rows["site_id"]        = sid
    rows["site_name"]      = site_meta["site_name"]
    rows["latitude"]       = site_meta["lat"]
    rows["longitude"]      = site_meta["lon"]
    rows["region"]         = site_meta["region"]
    rows["vendor"]         = site_meta["vendor"]

    # Infrastructure
    rows.update(bbu_cols(n, ts_arr, labels))
    rows.update(ru_ant_cols(n, labels))
    rows.update(bh_cols(n, labels))
    rows.update(power_cols(n, ts_arr, labels))
    rows.update(env_cols(n, ts_arr, labels))

    # Cell KPIs
    for c in range(1, N_CELLS+1):
        rows.update(cell_kpis(n, ts_arr, c, sid, labels))

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("\n" + "="*65)
    print("  RAN TRAINING DATA GENERATOR  (2015-01-01 → 2026-01-01)")
    print("="*65)

    timestamps = pd.date_range(START, END, freq=FREQ, inclusive="left")
    print(f"\nTimestamps: {len(timestamps):,} hourly rows × {len(SITES)} sites")
    print(f"Target failure rate: {TARGET_FAIL*100:.0f}%\n")

    frames = []
    for site in SITES:
        df = generate_site(site, timestamps)
        frames.append(df)

    full = pd.concat(frames, ignore_index=True)
    full = full.sort_values(["site_id", "timestamp"]).reset_index(drop=True)

    # ── Final stats ────────────────────────────────────────────────────────────
    total     = len(full)
    n_fail    = (full["scenario_label"] != "NORMAL").sum()
    fail_pct  = 100 * n_fail / total
    print(f"\n{'─'*65}")
    print(f"  Total rows      : {total:,}")
    print(f"  Failure rows    : {n_fail:,}  ({fail_pct:.2f}%)")
    print(f"  Normal rows     : {total-n_fail:,}  ({100-fail_pct:.2f}%)")
    print(f"  Columns         : {len(full.columns)}")
    print(f"  Date range      : {full['timestamp'].min()} → {full['timestamp'].max()}")
    print(f"\n  Scenario breakdown:")
    for sc, cnt in full["scenario_label"].value_counts().items():
        print(f"    {sc:<40s}  {cnt:7,}  ({100*cnt/total:.2f}%)")

    # ── Save ──────────────────────────────────────────────────────────────────
    full.to_parquet(OUTPUT, index=False, engine="pyarrow")
    mb = os.path.getsize(OUTPUT) / 1024 / 1024 if os.path.exists(OUTPUT) else 0
    print(f"\n  Saved → {OUTPUT}  ({mb:.1f} MB)")
    print("="*65 + "\n")
    return full


if __name__ == "__main__":
    df = main()


  RAN TRAINING DATA GENERATOR  (2015-01-01 → 2026-01-01)

Timestamps: 96,432 hourly rows × 3 sites
Target failure rate: 25%

  Generating SITE_001 (96,432 rows) …
    SITE_001: 24,108 failure rows (25.0%) / 96,432 total
  Generating SITE_002 (96,432 rows) …
    SITE_002: 24,108 failure rows (25.0%) / 96,432 total
  Generating SITE_003 (96,432 rows) …
    SITE_003: 24,108 failure rows (25.0%) / 96,432 total

─────────────────────────────────────────────────────────────────
  Total rows      : 289,296
  Failure rows    : 72,324  (25.00%)
  Normal rows     : 216,972  (75.00%)
  Columns         : 188
  Date range      : 2015-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00

  Scenario breakdown:
    NORMAL                                    216,972  (75.00%)
    ANT1_PHYSICAL_TILT_DRIFT                   15,328  (5.30%)
    GENERATOR_FUEL_DEPLETION                   11,652  (4.03%)
    ANT2_VSWR_ALARM                            10,748  (3.72%)
    CELL1_HIGH_BLER                          

In [7]:
"""
02_train_cell_model.py
=======================
RAN Cell Failure Prediction — Training on Cell Metrics Only

FEATURE SCOPE
-------------
Only the 27 per-cell RF/KPI metrics are used as raw inputs:
    status, op_state, active_users, connected_users,
    prb_utilization_percent, throughput_downlink/uplink_mbps,
    spectral_efficiency_bps_per_hz, rsrp_dbm, rsrq_db, sinr_db, cqi_avg,
    bler_downlink/uplink_percent, harq_retransmission_rate_percent,
    latency_downlink/uplink_ms, handover_attempts/success_rate/failures,
    rrc_connection_attempts/success_rate, erab_setup_success_rate,
    call_drop_rate_percent, abnormal_release_rate_percent,
    technology, bandwidth_mhz

Engineered from those:
    • Temporal flags from timestamp  (NOT the raw date)
    • Rolling mean/std/max/min at 3h, 6h, 12h, 24h
    • Lag values at 1h, 3h, 6h, 12h back
    • Rate-of-change (delta) at steps 1h and 3h

Label
-----
    binary:  1 = failure  (scenario_label != 'NORMAL')
             0 = normal

Output artefacts
----------------
    ran_cell_model.txt          LightGBM native (production, portable)
    ran_cell_model.pkl          joblib pickle   (sklearn-pipeline use)
    ran_cell_model_features.json  ordered feature list for inference
    ran_cell_training_report.png  evaluation charts

Designed for Google Colab — all installs included.
"""

# ── 0. Install ─────────────────────────────────────────────────────────────────
import subprocess, sys
def _pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
_pip("lightgbm","pyarrow","joblib","pandas","numpy","scikit-learn",
     "matplotlib","seaborn")

# ── 1. Imports ─────────────────────────────────────────────────────────────────
import os, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve,
    f1_score,
)

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

DATA_FILE      = "ran_training_data.parquet"
MODEL_TXT      = "ran_cell_model.txt"
MODEL_PKL      = "ran_cell_model.pkl"
FEATURES_JSON  = "ran_cell_model_features.json"
REPORT_PNG     = "ran_cell_training_report.png"

LABEL_COL      = "scenario_label"
NORMAL_VALUE   = "NORMAL"
N_CELLS        = 3
THRESHOLD      = 0.5
CV_SPLITS      = 5
EARLY_STOP     = 50

# ── LightGBM params ────────────────────────────────────────────────────────────
LGB_PARAMS = dict(
    objective        = "binary",
    boosting_type    = "gbdt",
    n_estimators     = 1000,      # upper bound; early stopping kicks in
    learning_rate    = 0.04,
    num_leaves       = 63,
    max_depth        = -1,
    min_child_samples= 40,
    subsample        = 0.8,
    subsample_freq   = 1,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    scale_pos_weight = 3,         # balances 25 % failure vs 75 % normal
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

# ── Cell metric columns (suffixes — will be resolved with cell_N_ prefix) ─────
CELL_METRIC_SUFFIXES = [
    "status", "op_state",
    "active_users", "connected_users",
    "prb_utilization_percent",
    "throughput_downlink_mbps", "throughput_uplink_mbps",
    "spectral_efficiency_bps_per_hz",
    "rsrp_dbm", "rsrq_db", "sinr_db", "cqi_avg",
    "bler_downlink_percent", "bler_uplink_percent",
    "harq_retransmission_rate_percent",
    "latency_downlink_ms", "latency_uplink_ms",
    "handover_attempts", "handover_success_rate_percent", "handover_failures",
    "rrc_connection_attempts", "rrc_success_rate_percent",
    "erab_setup_success_rate_percent",
    "call_drop_rate_percent", "abnormal_release_rate_percent",
    "technology", "bandwidth_mhz",
]

# KPI subsets used for time-series feature engineering
ROLL_KPIS = [
    "prb_utilization_percent",
    "throughput_downlink_mbps", "throughput_uplink_mbps",
    "sinr_db", "rsrp_dbm", "rsrq_db", "cqi_avg",
    "bler_downlink_percent", "bler_uplink_percent",
    "harq_retransmission_rate_percent",
    "latency_downlink_ms", "latency_uplink_ms",
    "call_drop_rate_percent", "handover_success_rate_percent",
]
LAG_KPIS = [
    "prb_utilization_percent", "sinr_db", "rsrp_dbm",
    "bler_downlink_percent", "call_drop_rate_percent",
]
DELTA_KPIS = [
    "prb_utilization_percent", "sinr_db", "bler_downlink_percent",
]

ROLL_WINDOWS = [3, 6, 12, 24]
LAG_STEPS    = [1, 3, 6, 12]
DELTA_STEPS  = [1, 3]

# Columns to drop before building X (meta / date / label)
DROP_COLS = {
    "timestamp", "message_id", "sequence_number",
    "site_id", "site_name", "scenario_label",
    "latitude", "longitude", "region", "vendor",
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_data(path: str) -> pd.DataFrame:
    print(f"  Loading {path} …")
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["site_id", "timestamp"]).reset_index(drop=True)
    print(f"  Rows: {len(df):,}   Columns: {len(df.columns)}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# SECTION C — WIDE → LONG  (one site-row → N_CELLS cell-rows)
# ══════════════════════════════════════════════════════════════════════════════

def wide_to_long(df: pd.DataFrame) -> pd.DataFrame:
    shared = ["timestamp", "site_id", LABEL_COL]
    shared = [c for c in shared if c in df.columns]

    frames = []
    for idx in range(1, N_CELLS + 1):
        prefix = f"cell_{idx}_"
        rename = {f"{prefix}{s}": s
                  for s in CELL_METRIC_SUFFIXES
                  if f"{prefix}{s}" in df.columns}
        tmp = df[shared + list(rename)].copy().rename(columns=rename)
        tmp["cell_id"] = np.int8(idx)
        frames.append(tmp)

    long = (pd.concat(frames, ignore_index=True)
              .sort_values(["site_id", "cell_id", "timestamp"])
              .reset_index(drop=True))
    print(f"  Wide→Long: {len(long):,} cell-rows")
    return long


# ══════════════════════════════════════════════════════════════════════════════
# SECTION D — LABEL
# ══════════════════════════════════════════════════════════════════════════════

def build_label(df: pd.DataFrame) -> pd.Series:
    y = (df[LABEL_COL] != NORMAL_VALUE).astype(np.int8)
    n1, n0 = y.sum(), (y == 0).sum()
    print(f"  Label: {n1:,} failures ({100*n1/len(y):.1f}%)  "
          f"| {n0:,} normal ({100*n0/len(y):.1f}%)")
    return y


# ══════════════════════════════════════════════════════════════════════════════
# SECTION E — FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════

# ── Status encoding ────────────────────────────────────────────────────────────
_STATUS_MAP = {
    "UP":1,"ACTIVE":1,"ON":1,"OK":1,"OPERATIONAL":1,"NORMAL":1,
    "DOWN":0,"INACTIVE":0,"OFF":0,"FAILED":0,"FAULT":0,
    "DEGRADED":2,"WARNING":2,"PARTIAL":2,
    "STANDBY":3,"IDLE":3,
}
_TECH_MAP = {"2G":0,"3G":1,"4G":2,"5G":3,"NR":3}


def encode_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.select_dtypes("object").columns:
        if col in DROP_COLS or col == LABEL_COL:
            continue
        if col == "technology":
            df[col] = df[col].map(_TECH_MAP).fillna(2).astype(np.int8)
        else:
            df[col] = df[col].map(_STATUS_MAP).fillna(0).astype(np.int8)
    return df


def add_temporal(df: pd.DataFrame) -> pd.DataFrame:
    ts = df["timestamp"]
    df = df.copy()
    df["hour_of_day"]  = ts.dt.hour.astype(np.int8)
    df["day_of_week"]  = ts.dt.dayofweek.astype(np.int8)
    df["day_of_month"] = ts.dt.day.astype(np.int8)
    df["month"]        = ts.dt.month.astype(np.int8)
    df["is_weekend"]   = (df["day_of_week"] >= 5).astype(np.int8)
    df["is_night"]     = ((df["hour_of_day"] >= 22) |
                          (df["hour_of_day"] < 6)).astype(np.int8)
    # ⚠  Raw timestamp intentionally NOT added as a feature
    return df


def add_rolling(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    g = df.groupby(["site_id", "cell_id"])
    for w in ROLL_WINDOWS:
        for kpi in ROLL_KPIS:
            if kpi not in df.columns:
                continue
            r = g[kpi].transform
            df[f"{kpi}_roll{w}_mean"] = r(lambda x,w=w: x.rolling(w,min_periods=1).mean())
            df[f"{kpi}_roll{w}_std"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).std().fillna(0))
            df[f"{kpi}_roll{w}_max"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).max())
            df[f"{kpi}_roll{w}_min"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).min())
    return df


def add_lags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    g = df.groupby(["site_id", "cell_id"])
    for kpi in LAG_KPIS:
        if kpi not in df.columns:
            continue
        for lag in LAG_STEPS:
            df[f"{kpi}_lag{lag}"] = g[kpi].transform(
                lambda x, l=lag: x.shift(l).bfill().fillna(x.mean()))
    return df


def add_deltas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    g = df.groupby(["site_id", "cell_id"])
    for kpi in DELTA_KPIS:
        if kpi not in df.columns:
            continue
        for step in DELTA_STEPS:
            df[f"{kpi}_delta{step}"] = g[kpi].transform(
                lambda x, s=step: x.diff(s).fillna(0))
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    print("  Encoding categoricals …")
    df = encode_cats(df)
    print("  Adding temporal features (NO raw date in model) …")
    df = add_temporal(df)
    print("  Adding rolling statistics …")
    df = add_rolling(df)
    print("  Adding lag features …")
    df = add_lags(df)
    print("  Adding delta features …")
    df = add_deltas(df)
    return df


def select_features(df: pd.DataFrame) -> list[str]:
    """Return final ordered feature column list."""
    exclude = DROP_COLS | {LABEL_COL}
    return [c for c in df.columns
            if c not in exclude and df[c].dtype != object]


# ══════════════════════════════════════════════════════════════════════════════
# SECTION F — TRAINING
# ══════════════════════════════════════════════════════════════════════════════

def train(X: pd.DataFrame, y: pd.Series):
    print(f"\n  Feature matrix : {X.shape[0]:,} rows × {X.shape[1]} features")

    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    cv_metrics = []

    print(f"  Time-series CV ({CV_SPLITS} folds) …")
    best_iters = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]

        m = lgb.LGBMClassifier(**LGB_PARAMS)
        m.fit(Xtr, ytr,
              eval_set=[(Xva, yva)],
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                         lgb.log_evaluation(period=-1)])

        prob = m.predict_proba(Xva)[:,1]
        pred = (prob >= THRESHOLD).astype(int)
        auc  = roc_auc_score(yva, prob)
        ap   = average_precision_score(yva, prob)
        f1   = f1_score(yva, pred)
        best_iters.append(m.best_iteration_)
        cv_metrics.append(dict(fold=fold, auc=auc, ap=ap, f1=f1,
                               best_iter=m.best_iteration_))
        print(f"    Fold {fold}: AUC={auc:.4f}  AP={ap:.4f}  "
              f"F1={f1:.4f}  trees={m.best_iteration_}")

    cv_df = pd.DataFrame(cv_metrics)
    print(f"\n  CV Summary:")
    print(f"    AUC : {cv_df.auc.mean():.4f} ± {cv_df.auc.std():.4f}")
    print(f"    AP  : {cv_df.ap.mean():.4f}  ± {cv_df.ap.std():.4f}")
    print(f"    F1  : {cv_df.f1.mean():.4f}  ± {cv_df.f1.std():.4f}")

    # Final model on full data with mean best iteration
    n_est = max(50, int(np.mean(best_iters)))
    print(f"\n  Training final model (n_estimators={n_est}) on full data …")
    final_params = {**LGB_PARAMS, "n_estimators": n_est}
    clf = lgb.LGBMClassifier(**final_params)
    clf.fit(X, y)
    print(f"  Done. Trees in final model: {clf.n_estimators_}")
    return clf, cv_df


# ══════════════════════════════════════════════════════════════════════════════
# SECTION G — EVALUATION & PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def evaluate(clf, X, y, cv_df, feat_cols, out_png):
    prob  = clf.predict_proba(X)[:,1]
    pred  = (prob >= THRESHOLD).astype(int)
    auc   = roc_auc_score(y, prob)
    ap    = average_precision_score(y, prob)
    cm    = confusion_matrix(y, pred)
    fpr, tpr, _ = roc_curve(y, prob)
    prec, rec, _= precision_recall_curve(y, prob)

    top_n = 30
    imp = pd.Series(clf.feature_importances_, index=feat_cols).nlargest(top_n)

    fig, axes = plt.subplots(2, 3, figsize=(22, 13))
    fig.suptitle("RAN Cell Model — Training Report", fontsize=15, fontweight="bold")
    plt.subplots_adjust(hspace=0.38, wspace=0.32)

    # 1. CV metrics per fold
    ax = axes[0, 0]
    x  = np.arange(1, len(cv_df)+1)
    w  = 0.25
    ax.bar(x-w, cv_df.auc, w, label="AUC",  color="#3b82f6", alpha=0.85)
    ax.bar(x,   cv_df.ap,  w, label="AP",   color="#f59e0b", alpha=0.85)
    ax.bar(x+w, cv_df.f1,  w, label="F1",   color="#10b981", alpha=0.85)
    for val, label, col in [(cv_df.auc.mean(),"#3b82f6","#3b82f6"),
                             (cv_df.ap.mean(), "#f59e0b","#f59e0b"),
                             (cv_df.f1.mean(), "#10b981","#10b981")]:
        ax.axhline(val, color=col, ls="--", lw=1, alpha=0.7)
    ax.set_title("CV Scores per Fold"); ax.set_xlabel("Fold"); ax.set_ylim(0, 1.05)
    ax.legend(); ax.grid(axis="y", alpha=0.3)

    # 2. ROC
    ax = axes[0, 1]
    ax.plot(fpr, tpr, "#3b82f6", lw=2, label=f"AUC = {auc:.4f}")
    ax.fill_between(fpr, tpr, alpha=0.1, color="#3b82f6")
    ax.plot([0,1],[0,1],"k--",lw=1)
    ax.set_title("ROC Curve"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(); ax.grid(alpha=0.3)

    # 3. PR Curve
    ax = axes[0, 2]
    ax.plot(rec, prec, "#10b981", lw=2, label=f"AP = {ap:.4f}")
    ax.fill_between(rec, prec, alpha=0.1, color="#10b981")
    ax.set_title("Precision-Recall Curve")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.legend(); ax.grid(alpha=0.3)

    # 4. Confusion matrix
    ax = axes[1, 0]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Normal","Failure"],
                yticklabels=["Normal","Failure"])
    ax.set_title("Confusion Matrix (t=0.5)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")

    # 5. Score distribution
    ax = axes[1, 1]
    ax.hist(prob[y==0], bins=50, alpha=0.6, color="#3b82f6",
            density=True, label="Normal (0)")
    ax.hist(prob[y==1], bins=50, alpha=0.6, color="#ef4444",
            density=True, label="Failure (1)")
    ax.axvline(THRESHOLD, color="k", ls="--", lw=1.3, label="Threshold")
    ax.set_title("Predicted Probability Distribution")
    ax.set_xlabel("P(failure)"); ax.set_ylabel("Density")
    ax.legend(); ax.grid(alpha=0.3)

    # 6. Top features
    ax = axes[1, 2]
    colors = ["#ef4444" if v == imp.max() else "#6366f1"
              for v in imp.values]
    imp[::-1].plot.barh(ax=ax, color=colors[::-1], alpha=0.85)
    ax.set_title(f"Top {top_n} Feature Importances")
    ax.set_xlabel("Split Count"); ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="x", alpha=0.3)

    plt.savefig(out_png, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Report saved → {out_png}")

    print("\n  Classification Report (threshold=0.5, full train set):")
    print(classification_report(y, pred, target_names=["Normal","Failure"],
                                 digits=4))


# ══════════════════════════════════════════════════════════════════════════════
# SECTION H — SAVE PRODUCTION ARTEFACTS
# ══════════════════════════════════════════════════════════════════════════════

def save_artefacts(clf, feat_cols):
    # 1. LightGBM native text
    clf.booster_.save_model(MODEL_TXT)
    print(f"  LightGBM native → {MODEL_TXT}  "
          f"({Path(MODEL_TXT).stat().st_size/1024:.1f} KB)")

    # 2. joblib pickle
    joblib.dump(clf, MODEL_PKL, compress=3)
    print(f"  joblib pickle   → {MODEL_PKL}  "
          f"({Path(MODEL_PKL).stat().st_size/1024:.1f} KB)")

    # 3. Feature manifest  (used by the inference script)
    meta = {
        "feature_columns":      feat_cols,
        "n_features":           len(feat_cols),
        "cell_metric_suffixes": CELL_METRIC_SUFFIXES,
        "roll_kpis":            ROLL_KPIS,
        "lag_kpis":             LAG_KPIS,
        "delta_kpis":           DELTA_KPIS,
        "roll_windows":         ROLL_WINDOWS,
        "lag_steps":            LAG_STEPS,
        "delta_steps":          DELTA_STEPS,
        "n_cells":              N_CELLS,
        "label_col":            LABEL_COL,
        "normal_value":         NORMAL_VALUE,
        "threshold":            THRESHOLD,
        "status_map":           _STATUS_MAP,
        "tech_map":             _TECH_MAP,
    }
    with open(FEATURES_JSON, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  Feature manifest→ {FEATURES_JSON}")
    print("\n  ✅  All production artefacts saved.")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    t0 = time.time()
    print("\n" + "="*65)
    print("  RAN CELL FAILURE MODEL — TRAINING PIPELINE")
    print("="*65)

    print("\n[1/6] Loading data …")
    raw = load_data(DATA_FILE)

    print("\n[2/6] Wide → Long …")
    long = wide_to_long(raw)

    print("\n[3/6] Building label …")
    y = build_label(long)

    print("\n[4/6] Engineering features …")
    feat_df = engineer_features(long)
    feat_cols = select_features(feat_df)
    X = feat_df[feat_cols].fillna(0).astype(np.float32)
    print(f"  Final feature matrix: {X.shape}")

    print("\n[5/6] Training …")
    clf, cv_df = train(X, y)

    print("\n[6/6] Evaluating & saving …")
    evaluate(clf, X, y, cv_df, feat_cols, REPORT_PNG)
    save_artefacts(clf, feat_cols)

    print(f"\n{'='*65}")
    print(f"  Done in {time.time()-t0:.1f}s")
    print(f"  Models   : {MODEL_TXT}  |  {MODEL_PKL}")
    print(f"  Manifest : {FEATURES_JSON}")
    print(f"  Report   : {REPORT_PNG}")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()



  RAN CELL FAILURE MODEL — TRAINING PIPELINE

[1/6] Loading data …
  Loading ran_training_data.parquet …
  Rows: 289,296   Columns: 188

[2/6] Wide → Long …
  Wide→Long: 867,888 cell-rows

[3/6] Building label …
  Label: 216,972 failures (25.0%)  | 650,916 normal (75.0%)

[4/6] Engineering features …
  Encoding categoricals …
  Adding temporal features (NO raw date in model) …
  Adding rolling statistics …
  Adding lag features …
  Adding delta features …
  Final feature matrix: (867888, 284)

[5/6] Training …

  Feature matrix : 867,888 rows × 284 features
  Time-series CV (5 folds) …
    Fold 1: AUC=0.7773  AP=0.7157  F1=0.6608  trees=246
    Fold 2: AUC=0.8248  AP=0.7931  F1=0.7798  trees=705
    Fold 3: AUC=0.7365  AP=0.6662  F1=0.6071  trees=774
    Fold 4: AUC=0.8187  AP=0.7912  F1=0.7709  trees=1000
    Fold 5: AUC=0.7460  AP=0.6754  F1=0.6129  trees=1000

  CV Summary:
    AUC : 0.7807 ± 0.0405
    AP  : 0.7283  ± 0.0612
    F1  : 0.6863  ± 0.0840

  Training final model (n_es

In [8]:
"""
"""
03_predict.py
==============
RAN Cell Failure Prediction — Production-Ready Inference

USAGE
-----
    python 03_predict.py \
        --data   new_site_data.parquet   \
        --model  ran_cell_model.txt      \
        --meta   ran_cell_model_features.json \
        --output predictions.csv

    # or import as a library:
    from 03_predict import RANCellPredictor
    predictor = RANCellPredictor("ran_cell_model.txt",
                                  "ran_cell_model_features.json")
    results = predictor.predict("new_site_data.parquet")

INPUT FORMAT
------------
A .parquet file with the same wide-format schema produced by the data
generator (one row per site per hour, columns: cell_1_*, cell_2_*, cell_3_*).
The file does NOT need to contain scenario_label.

OUTPUT FORMAT
-------------
A CSV with one row per (site, cell, timestamp):
    timestamp, site_id, cell_id, failure_probability,
    predicted_failure, risk_level

RISK LEVELS
-----------
    LOW       P < 0.30
    MEDIUM    0.30 ≤ P < 0.55
    HIGH      0.55 ≤ P < 0.75
    CRITICAL  P ≥ 0.75

Designed for Google Colab — all installs included.
"""

# ── 0. Install ─────────────────────────────────────────────────────────────────
import subprocess, sys
def _pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
_pip("lightgbm","pyarrow","joblib","pandas","numpy")

# ── 1. Imports ─────────────────────────────────────────────────────────────────
import argparse, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DEFAULTS  (override via CLI args or RANCellPredictor constructor)
# ══════════════════════════════════════════════════════════════════════════════
DEFAULT_DATA    = "ran_test_data.parquet"   # any site parquet
DEFAULT_MODEL   = "ran_cell_model.txt"
DEFAULT_META    = "ran_cell_model_features.json"
DEFAULT_OUTPUT  = "ran_predictions.csv"
DEFAULT_THRESH  = 0.5                           # overridden by meta JSON if present

# ══════════════════════════════════════════════════════════════════════════════
# STATIC ENCODINGS  (must match training script exactly)
# ══════════════════════════════════════════════════════════════════════════════
_STATUS_MAP = {
    "UP":1,"ACTIVE":1,"ON":1,"OK":1,"OPERATIONAL":1,"NORMAL":1,
    "DOWN":0,"INACTIVE":0,"OFF":0,"FAILED":0,"FAULT":0,
    "DEGRADED":2,"WARNING":2,"PARTIAL":2,
    "STANDBY":3,"IDLE":3,
}
_TECH_MAP = {"2G":0,"3G":1,"4G":2,"5G":3,"NR":3}

# Risk level bins
_RISK_BINS   = [0.0, 0.30, 0.55, 0.75, 1.001]
_RISK_LABELS = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]


# ══════════════════════════════════════════════════════════════════════════════
# CLASS: RANCellPredictor
# ══════════════════════════════════════════════════════════════════════════════

class RANCellPredictor:
    """
    Production-ready inference class for RAN cell failure prediction.

    Parameters
    ----------
    model_path  : str   Path to ran_cell_model.txt  OR  .pkl
    meta_path   : str   Path to ran_cell_model_features.json
    threshold   : float Decision threshold (default from meta or 0.5)
    """

    def __init__(self,
                 model_path: str = DEFAULT_MODEL,
                 meta_path:  str = DEFAULT_META,
                 threshold:  float | None = None):

        self.model_path = Path(model_path)
        self.meta_path  = Path(meta_path)

        # ── Load feature manifest ──────────────────────────────────────────────
        if not self.meta_path.exists():
            raise FileNotFoundError(
                f"Feature manifest not found: {meta_path}\n"
                "Run 02_train_cell_model.py first to generate it."
            )
        with open(self.meta_path) as f:
            self.meta = json.load(f)

        self.feature_columns      = self.meta["feature_columns"]
        self.cell_metric_suffixes = self.meta["cell_metric_suffixes"]
        self.roll_kpis            = self.meta["roll_kpis"]
        self.lag_kpis             = self.meta["lag_kpis"]
        self.delta_kpis           = self.meta["delta_kpis"]
        self.roll_windows         = self.meta["roll_windows"]
        self.lag_steps            = self.meta["lag_steps"]
        self.delta_steps          = self.meta["delta_steps"]
        self.n_cells              = self.meta.get("n_cells", 3)
        self.threshold            = threshold or self.meta.get("threshold", DEFAULT_THRESH)

        # ── Load model ────────────────────────────────────────────────────────
        if not self.model_path.exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")

        if self.model_path.suffix == ".pkl":
            obj = joblib.load(self.model_path)
            # unwrap LGBMClassifier → Booster for uniform .predict() interface
            self.booster = obj.booster_ if hasattr(obj, "booster_") else obj
            self._is_classifier = hasattr(obj, "predict_proba")
            self._clf = obj
        else:
            self.booster = lgb.Booster(model_file=str(self.model_path))
            self._is_classifier = False
            self._clf = None

        n_model_feats = (self.booster.num_feature()
                         if not self._is_classifier
                         else self._clf.n_features_in_)
        print(f"[MODEL] Loaded: {self.model_path.name}  "
              f"({n_model_feats} features  |  "
              f"threshold={self.threshold})")

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────────────────────

    def predict(self,
                data_source: "str | pd.DataFrame",
                output_path: str | None = None) -> pd.DataFrame:
        """
        Full pipeline: load → preprocess → predict → return DataFrame.

        Parameters
        ----------
        data_source : path to parquet file  OR  already-loaded wide DataFrame
        output_path : if given, saves predictions to CSV at this path

        Returns
        -------
        pd.DataFrame with columns:
            timestamp, site_id, cell_id, failure_probability,
            predicted_failure, risk_level
        """
        raw = self._load(data_source)
        long_df = self._wide_to_long(raw)
        feat_df = self._engineer_features(long_df)
        X       = self._align_features(feat_df)
        prob    = self._score(X)
        results = self._assemble_output(feat_df, prob)

        if output_path:
            results.to_csv(output_path, index=False)
            print(f"[OUTPUT] Saved {len(results):,} rows → {output_path}")

        self._print_summary(results)
        return results

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 1: LOAD
    # ──────────────────────────────────────────────────────────────────────────

    def _load(self, source) -> pd.DataFrame:
        if isinstance(source, pd.DataFrame):
            df = source.copy()
        else:
            df = pd.read_parquet(source)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values(["site_id", "timestamp"]).reset_index(drop=True)
        print(f"[LOAD]  {len(df):,} site-rows × {len(df.columns)} columns")
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 2: WIDE → LONG
    # ──────────────────────────────────────────────────────────────────────────

    def _wide_to_long(self, df: pd.DataFrame) -> pd.DataFrame:
        # Shared columns: id + anything that is NOT a cell_N_ column
        shared = ["timestamp", "site_id"]
        shared = [c for c in shared if c in df.columns]

        frames = []
        for idx in range(1, self.n_cells + 1):
            prefix = f"cell_{idx}_"
            rename = {f"{prefix}{s}": s
                      for s in self.cell_metric_suffixes
                      if f"{prefix}{s}" in df.columns}
            tmp = df[shared + list(rename)].copy().rename(columns=rename)
            tmp["cell_id"] = np.int8(idx)
            frames.append(tmp)

        long = (pd.concat(frames, ignore_index=True)
                  .sort_values(["site_id", "cell_id", "timestamp"])
                  .reset_index(drop=True))
        print(f"[TRANSFORM] Wide→Long: {len(long):,} cell-rows")
        return long

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 3: FEATURE ENGINEERING  (must be identical to training)
    # ──────────────────────────────────────────────────────────────────────────

    def _encode_cats(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col in df.select_dtypes("object").columns:
            if col in ("site_id", "timestamp"):
                continue
            if col == "technology":
                df[col] = df[col].map(_TECH_MAP).fillna(2).astype(np.int8)
            else:
                df[col] = df[col].map(_STATUS_MAP).fillna(0).astype(np.int8)
        return df

    def _add_temporal(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        ts = df["timestamp"]
        df["hour_of_day"]  = ts.dt.hour.astype(np.int8)
        df["day_of_week"]  = ts.dt.dayofweek.astype(np.int8)
        df["day_of_month"] = ts.dt.day.astype(np.int8)
        df["month"]        = ts.dt.month.astype(np.int8)
        df["is_weekend"]   = (df["day_of_week"] >= 5).astype(np.int8)
        df["is_night"]     = ((df["hour_of_day"] >= 22) |
                               (df["hour_of_day"] < 6)).astype(np.int8)
        return df

    def _add_rolling(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby(["site_id", "cell_id"])
        for w in self.roll_windows:
            for kpi in self.roll_kpis:
                if kpi not in df.columns:
                    continue
                r = g[kpi].transform
                df[f"{kpi}_roll{w}_mean"] = r(lambda x,w=w: x.rolling(w,min_periods=1).mean())
                df[f"{kpi}_roll{w}_std"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).std().fillna(0))
                df[f"{kpi}_roll{w}_max"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).max())
                df[f"{kpi}_roll{w}_min"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).min())
        return df

    def _add_lags(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby(["site_id", "cell_id"])
        for kpi in self.lag_kpis:
            if kpi not in df.columns:
                continue
            for lag in self.lag_steps:
                df[f"{kpi}_lag{lag}"] = g[kpi].transform(
                    lambda x, l=lag: x.shift(l).bfill().fillna(x.mean()))
        return df

    def _add_deltas(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby(["site_id", "cell_id"])
        for kpi in self.delta_kpis:
            if kpi not in df.columns:
                continue
            for step in self.delta_steps:
                df[f"{kpi}_delta{step}"] = g[kpi].transform(
                    lambda x, s=step: x.diff(s).fillna(0))
        return df

    def _engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[FEATURES] Encoding → temporal → rolling → lag → delta …")
        df = self._encode_cats(df)
        df = self._add_temporal(df)
        df = self._add_rolling(df)
        df = self._add_lags(df)
        df = self._add_deltas(df)
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 4: ALIGN TO EXACT TRAINING FEATURE ORDER
    # ──────────────────────────────────────────────────────────────────────────

    def _align_features(self, df: pd.DataFrame) -> pd.DataFrame:
        missing = [f for f in self.feature_columns if f not in df.columns]
        if missing:
            print(f"[ALIGN]  {len(missing)} features missing → filled with 0")
            for f in missing:
                df[f] = 0.0
        X = df[self.feature_columns].fillna(0).astype(np.float32)
        print(f"[ALIGN]  Feature matrix: {X.shape}")
        return X

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 5: SCORE
    # ──────────────────────────────────────────────────────────────────────────

    def _score(self, X: pd.DataFrame) -> np.ndarray:
        if self._is_classifier and self._clf is not None:
            prob = self._clf.predict_proba(X.values)[:, 1]
        else:
            prob = self.booster.predict(X.values)
        print(f"[SCORE]  Scored {len(prob):,} rows")
        return prob.astype(np.float32)

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 6: ASSEMBLE OUTPUT
    # ──────────────────────────────────────────────────────────────────────────

    def _assemble_output(self, df: pd.DataFrame,
                          prob: np.ndarray) -> pd.DataFrame:
        out = pd.DataFrame({
            "timestamp":           df["timestamp"],
            "site_id":             df["site_id"],
            "cell_id":             df["cell_id"],
            "failure_probability": np.round(prob, 4),
            "predicted_failure":   (prob >= self.threshold).astype(np.int8),
            "risk_level": pd.cut(
                prob,
                bins=_RISK_BINS,
                labels=_RISK_LABELS,
                right=False
            ),
        })
        return out

    # ──────────────────────────────────────────────────────────────────────────
    # SUMMARY
    # ──────────────────────────────────────────────────────────────────────────

    @staticmethod
    def _print_summary(results: pd.DataFrame):
        total = len(results)
        n_fail = results["predicted_failure"].sum()
        print(f"\n{'─'*55}")
        print(f"  Total cell-hours scored : {total:,}")
        print(f"  Predicted failures      : {n_fail:,}  "
              f"({100*n_fail/total:.1f}%)")
        print(f"\n  By cell:")
        for cid, grp in results.groupby("cell_id"):
            f = grp["predicted_failure"].sum()
            print(f"    Cell {cid}: {f:5,} / {len(grp):,}  "
                  f"({100*f/len(grp):.1f}%)")
        print(f"\n  Risk breakdown:")
        for lv in _RISK_LABELS:
            cnt = (results["risk_level"] == lv).sum()
            print(f"    {lv:<10s}: {cnt:7,}  ({100*cnt/total:.1f}%)")
        print(f"{'─'*55}\n")


# ══════════════════════════════════════════════════════════════════════════════
# CLI ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def _parse_args():
    p = argparse.ArgumentParser(
        description="RAN Cell Failure — Production Inference")
    p.add_argument("--data",   default=DEFAULT_DATA,
                   help="Input .parquet file (wide format)")
    p.add_argument("--model",  default=DEFAULT_MODEL,
                   help=".txt or .pkl model file")
    p.add_argument("--meta",   default=DEFAULT_META,
                   help="Feature manifest JSON from training")
    p.add_argument("--output", default=DEFAULT_OUTPUT,
                   help="Output CSV path")
    p.add_argument("--threshold", type=float, default=None,
                   help="Decision threshold (overrides meta JSON)")
    return p.parse_args()


def main():
    args  = _parse_args()
    pred  = RANCellPredictor(args.model, args.meta, args.threshold)
    results = pred.predict(args.data, args.output)
    print(results.head(20).to_string(index=False))


# ── Colab convenience: run directly without CLI ───────────────────────────────
if __name__ == "__main__":
    import sys
    if len(sys.argv) == 1:
        # No CLI args: use defaults (works in Colab)
        print("Running with default paths (Colab mode) …\n")
        predictor = RANCellPredictor(DEFAULT_MODEL, DEFAULT_META)
        results   = predictor.predict(DEFAULT_DATA, DEFAULT_OUTPUT)
        print(results.head(30).to_string(index=False))
    else:
        main()
"""


usage: colab_kernel_launcher.py [-h] [--data DATA] [--model MODEL]
                                [--meta META] [--output OUTPUT]
                                [--threshold THRESHOLD]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-6518db16-d7b9-4140-8472-f8fb3242eafc.json


SystemExit: 2

In [17]:
"""
03_predict.py
==============
RAN Cell Failure Prediction — Production-Ready Inference

USAGE
-----
    python 03_predict.py \
        --data   new_site_data.parquet   \
        --model  ran_cell_model.txt      \
        --meta   ran_cell_model_features.json \
        --output predictions.csv

    # or import as a library:
    from 03_predict import RANCellPredictor
    predictor = RANCellPredictor("ran_cell_model.txt",
                                  "ran_cell_model_features.json")
    results = predictor.predict("new_site_data.parquet")

INPUT FORMAT
------------
A .parquet file with the same wide-format schema produced by the data
generator (one row per site per hour, columns: cell_1_*, cell_2_*, cell_3_*).
The file does NOT need to contain scenario_label.

OUTPUT FORMAT
-------------
A CSV with one row per (site, cell, timestamp):
    timestamp, site_id, cell_id, failure_probability,
    predicted_failure, risk_level

RISK LEVELS
-----------
    LOW       P < 0.30
    MEDIUM    0.30 ≤ P < 0.55
    HIGH      0.55 ≤ P < 0.75
    CRITICAL  P ≥ 0.75

Designed for Google Colab — all installs included.
"""

# ── 0. Install ─────────────────────────────────────────────────────────────────
import subprocess, sys
def _pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
_pip("lightgbm","pyarrow","joblib","pandas","numpy")

# ── 1. Imports ─────────────────────────────────────────────────────────────────
import argparse, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DEFAULTS  (override via CLI args or RANCellPredictor constructor)
# ══════════════════════════════════════════════════════════════════════════════
DEFAULT_DATA    = "my_test_2.parquet"   # any site parquet
DEFAULT_MODEL   = "ran_cell_model.txt"
DEFAULT_META    = "ran_cell_model_features.json"
DEFAULT_OUTPUT  = "ran_predictions.csv"
DEFAULT_THRESH  = 0.5                           # overridden by meta JSON if present

# ══════════════════════════════════════════════════════════════════════════════
# STATIC ENCODINGS  (must match training script exactly)
# ══════════════════════════════════════════════════════════════════════════════
_STATUS_MAP = {
    "UP":1,"ACTIVE":1,"ON":1,"OK":1,"OPERATIONAL":1,"NORMAL":1,
    "DOWN":0,"INACTIVE":0,"OFF":0,"FAILED":0,"FAULT":0,
    "DEGRADED":2,"WARNING":2,"PARTIAL":2,
    "STANDBY":3,"IDLE":3,
}
_TECH_MAP = {"2G":0,"3G":1,"4G":2,"5G":3,"NR":3}

# Risk level bins
_RISK_BINS   = [0.0, 0.30, 0.55, 0.75, 1.001]
_RISK_LABELS = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]


# ══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK / COLAB DETECTION
# ══════════════════════════════════════════════════════════════════════════════

def _is_notebook() -> bool:
    """Return True when running inside a Jupyter / Colab kernel."""
    try:
        shell = get_ipython().__class__.__name__   # type: ignore[name-defined]
        return shell in ("ZMQInteractiveShell", "Shell", "google.colab._shell")
    except NameError:
        return False


# ══════════════════════════════════════════════════════════════════════════════
# CLASS: RANCellPredictor
# ══════════════════════════════════════════════════════════════════════════════

class RANCellPredictor:
    """
    Production-ready inference class for RAN cell failure prediction.

    Parameters
    ----------
    model_path  : str   Path to ran_cell_model.txt  OR  .pkl
    meta_path   : str   Path to ran_cell_model_features.json
    threshold   : float Decision threshold (default from meta or 0.5)
    """

    def __init__(self,
                 model_path: str = DEFAULT_MODEL,
                 meta_path:  str = DEFAULT_META,
                 threshold:  float | None = None):

        self.model_path = Path(model_path)
        self.meta_path  = Path(meta_path)

        # ── Load feature manifest ──────────────────────────────────────────────
        if not self.meta_path.exists():
            raise FileNotFoundError(
                f"Feature manifest not found: {meta_path}\n"
                "Run 02_train_cell_model.py first to generate it."
            )
        with open(self.meta_path) as f:
            self.meta = json.load(f)

        self.feature_columns      = self.meta["feature_columns"]
        self.cell_metric_suffixes = self.meta["cell_metric_suffixes"]
        self.roll_kpis            = self.meta["roll_kpis"]
        self.lag_kpis             = self.meta["lag_kpis"]
        self.delta_kpis           = self.meta["delta_kpis"]
        self.roll_windows         = self.meta["roll_windows"]
        self.lag_steps            = self.meta["lag_steps"]
        self.delta_steps          = self.meta["delta_steps"]
        self.n_cells              = self.meta.get("n_cells", 3)
        self.threshold            = threshold or self.meta.get("threshold", DEFAULT_THRESH)

        # ── Load model ────────────────────────────────────────────────────────
        if not self.model_path.exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")

        if self.model_path.suffix == ".pkl":
            obj = joblib.load(self.model_path)
            # unwrap LGBMClassifier → Booster for uniform .predict() interface
            self.booster = obj.booster_ if hasattr(obj, "booster_") else obj
            self._is_classifier = hasattr(obj, "predict_proba")
            self._clf = obj
        else:
            self.booster = lgb.Booster(model_file=str(self.model_path))
            self._is_classifier = False
            self._clf = None

        n_model_feats = (self.booster.num_feature()
                         if not self._is_classifier
                         else self._clf.n_features_in_)
        print(f"[MODEL] Loaded: {self.model_path.name}  "
              f"({n_model_feats} features  |  "
              f"threshold={self.threshold})")

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────────────────────

    def predict(self,
                data_source: "str | pd.DataFrame",
                output_path: str | None = None) -> pd.DataFrame:
        """
        Full pipeline: load → preprocess → predict → return DataFrame.

        Parameters
        ----------
        data_source : path to parquet file  OR  already-loaded wide DataFrame
        output_path : if given, saves predictions to CSV at this path

        Returns
        -------
        pd.DataFrame with columns:
            timestamp, site_id, cell_id, failure_probability,
            predicted_failure, risk_level
        """
        raw = self._load(data_source)
        long_df = self._wide_to_long(raw)
        feat_df = self._engineer_features(long_df)
        X       = self._align_features(feat_df)
        prob    = self._score(X)
        results = self._assemble_output(feat_df, prob)

        if output_path:
            results.to_csv(output_path, index=False)
            print(f"[OUTPUT] Saved {len(results):,} rows → {output_path}")

        self._print_summary(results)
        return results

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 1: LOAD
    # ──────────────────────────────────────────────────────────────────────────

    @staticmethod
    def _sniff_and_read(path: Path) -> pd.DataFrame:
        """
        Detect the real file format by magic bytes and extension, then read it.
        Handles files that are named .parquet but are actually CSV/JSON/Excel,
        or genuine Parquet files with any extension.

        Priority:
          1. Magic-byte detection  (Parquet PAR1, Gzip \x1f\x8b, ZIP PK)
          2. Extension fallback    (.csv, .tsv, .json, .xlsx, .xls)
          3. Plain-text heuristic  (try CSV as last resort)
        """
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Data file not found: {path}")

        # Read first 8 bytes for magic detection
        with open(path, "rb") as fh:
            magic = fh.read(8)

        # ── Parquet: starts with PAR1 (or ends with PAR1 — check both) ────────
        is_parquet_magic = magic[:4] == b"PAR1"
        if not is_parquet_magic:
            # Parquet footer magic is at the end of the file
            with open(path, "rb") as fh:
                fh.seek(-4, 2)
                is_parquet_magic = fh.read(4) == b"PAR1"

        if is_parquet_magic:
            print(f"[LOAD]  Detected format: Parquet")
            return pd.read_parquet(path)

        # ── Gzip-compressed (could be .parquet.gz or .csv.gz) ─────────────────
        if magic[:2] == b"\x1f\x8b":
            ext = "".join(path.suffixes).lower()
            if ".csv" in ext or ".tsv" in ext:
                print(f"[LOAD]  Detected format: gzip CSV")
                return pd.read_csv(path, compression="gzip")
            # Assume gzip-parquet
            print(f"[LOAD]  Detected format: gzip Parquet")
            return pd.read_parquet(path)

        # ── ZIP / xlsx ─────────────────────────────────────────────────────────
        if magic[:2] == b"PK":
            print(f"[LOAD]  Detected format: Excel/ZIP")
            return pd.read_excel(path)

        # ── Extension-based fallback ───────────────────────────────────────────
        ext = path.suffix.lower()
        if ext in (".csv", ".tsv"):
            sep = "\t" if ext == ".tsv" else ","
            print(f"[LOAD]  Detected format: {'TSV' if ext == '.tsv' else 'CSV'}")
            return pd.read_csv(path, sep=sep)
        if ext in (".json", ".jsonl"):
            print(f"[LOAD]  Detected format: JSON")
            return pd.read_json(path, lines=(ext == ".jsonl"))
        if ext in (".xlsx", ".xls"):
            print(f"[LOAD]  Detected format: Excel")
            return pd.read_excel(path)

        # ── Last resort: try CSV (plain text) ─────────────────────────────────
        try:
            print(f"[LOAD]  Detected format: CSV (fallback)")
            return pd.read_csv(path)
        except Exception:
            pass

        raise ValueError(
            f"Cannot determine file format for: {path}\n"
            "Supported formats: Parquet, CSV, TSV, JSON, Excel (.xlsx/.xls), "
            "or gzip-compressed variants."
        )

    def _load(self, source) -> pd.DataFrame:
        if isinstance(source, pd.DataFrame):
            df = source.copy()
        else:
            df = self._sniff_and_read(Path(source))
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values(["site_id", "timestamp"]).reset_index(drop=True)
        print(f"[LOAD]  {len(df):,} site-rows × {len(df.columns)} columns")
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 2: WIDE → LONG
    # ──────────────────────────────────────────────────────────────────────────

    def _wide_to_long(self, df: pd.DataFrame) -> pd.DataFrame:
        # Shared columns: id + anything that is NOT a cell_N_ column
        shared = ["timestamp", "site_id"]
        shared = [c for c in shared if c in df.columns]

        frames = []
        for idx in range(1, self.n_cells + 1):
            prefix = f"cell_{idx}_"
            rename = {f"{prefix}{s}": s
                      for s in self.cell_metric_suffixes
                      if f"{prefix}{s}" in df.columns}
            tmp = df[shared + list(rename)].copy().rename(columns=rename)
            tmp["cell_id"] = np.int8(idx)
            frames.append(tmp)

        long = (pd.concat(frames, ignore_index=True)
                  .sort_values(["site_id", "cell_id", "timestamp"])
                  .reset_index(drop=True))
        print(f"[TRANSFORM] Wide→Long: {len(long):,} cell-rows")
        return long

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 3: FEATURE ENGINEERING  (must be identical to training)
    # ──────────────────────────────────────────────────────────────────────────

    def _encode_cats(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col in df.select_dtypes("object").columns:
            if col in ("site_id", "timestamp"):
                continue
            if col == "technology":
                df[col] = df[col].map(_TECH_MAP).fillna(2).astype(np.int8)
            else:
                df[col] = df[col].map(_STATUS_MAP).fillna(0).astype(np.int8)
        return df

    def _add_temporal(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        ts = df["timestamp"]
        df["hour_of_day"]  = ts.dt.hour.astype(np.int8)
        df["day_of_week"]  = ts.dt.dayofweek.astype(np.int8)
        df["day_of_month"] = ts.dt.day.astype(np.int8)
        df["month"]        = ts.dt.month.astype(np.int8)
        df["is_weekend"]   = (df["day_of_week"] >= 5).astype(np.int8)
        df["is_night"]     = ((df["hour_of_day"] >= 22) |
                               (df["hour_of_day"] < 6)).astype(np.int8)
        return df

    def _add_rolling(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby(["site_id", "cell_id"])
        for w in self.roll_windows:
            for kpi in self.roll_kpis:
                if kpi not in df.columns:
                    continue
                r = g[kpi].transform
                df[f"{kpi}_roll{w}_mean"] = r(lambda x,w=w: x.rolling(w,min_periods=1).mean())
                df[f"{kpi}_roll{w}_std"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).std().fillna(0))
                df[f"{kpi}_roll{w}_max"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).max())
                df[f"{kpi}_roll{w}_min"]  = r(lambda x,w=w: x.rolling(w,min_periods=1).min())
        return df

    def _add_lags(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby(["site_id", "cell_id"])
        for kpi in self.lag_kpis:
            if kpi not in df.columns:
                continue
            for lag in self.lag_steps:
                df[f"{kpi}_lag{lag}"] = g[kpi].transform(
                    lambda x, l=lag: x.shift(l).bfill().fillna(x.mean()))
        return df

    def _add_deltas(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby(["site_id", "cell_id"])
        for kpi in self.delta_kpis:
            if kpi not in df.columns:
                continue
            for step in self.delta_steps:
                df[f"{kpi}_delta{step}"] = g[kpi].transform(
                    lambda x, s=step: x.diff(s).fillna(0))
        return df

    def _engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[FEATURES] Encoding → temporal → rolling → lag → delta …")
        df = self._encode_cats(df)
        df = self._add_temporal(df)
        df = self._add_rolling(df)
        df = self._add_lags(df)
        df = self._add_deltas(df)
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 4: ALIGN TO EXACT TRAINING FEATURE ORDER
    # ──────────────────────────────────────────────────────────────────────────

    def _align_features(self, df: pd.DataFrame) -> pd.DataFrame:
        missing = [f for f in self.feature_columns if f not in df.columns]
        if missing:
            print(f"[ALIGN]  {len(missing)} features missing → filled with 0")
            for f in missing:
                df[f] = 0.0
        X = df[self.feature_columns].fillna(0).astype(np.float32)
        print(f"[ALIGN]  Feature matrix: {X.shape}")
        return X

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 5: SCORE
    # ──────────────────────────────────────────────────────────────────────────

    def _score(self, X: pd.DataFrame) -> np.ndarray:
        if self._is_classifier and self._clf is not None:
            prob = self._clf.predict_proba(X.values)[:, 1]
        else:
            prob = self.booster.predict(X.values)
        print(f"[SCORE]  Scored {len(prob):,} rows")
        return prob.astype(np.float32)

    # ──────────────────────────────────────────────────────────────────────────
    # STEP 6: ASSEMBLE OUTPUT
    # ──────────────────────────────────────────────────────────────────────────

    def _assemble_output(self, df: pd.DataFrame,
                          prob: np.ndarray) -> pd.DataFrame:
        out = pd.DataFrame({
            "timestamp":           df["timestamp"],
            "site_id":             df["site_id"],
            "cell_id":             df["cell_id"],
            "failure_probability": np.round(prob, 4),
            "predicted_failure":   (prob >= self.threshold).astype(np.int8),
            "risk_level": pd.cut(
                prob,
                bins=_RISK_BINS,
                labels=_RISK_LABELS,
                right=False
            ),
        })
        return out

    # ──────────────────────────────────────────────────────────────────────────
    # SUMMARY
    # ──────────────────────────────────────────────────────────────────────────

    @staticmethod
    def _print_summary(results: pd.DataFrame):
        total = len(results)
        n_fail = results["predicted_failure"].sum()
        print(f"\n{'─'*55}")
        print(f"  Total cell-hours scored : {total:,}")
        print(f"  Predicted failures      : {n_fail:,}  "
              f"({100*n_fail/total:.1f}%)")
        print(f"\n  By cell:")
        for cid, grp in results.groupby("cell_id"):
            f = grp["predicted_failure"].sum()
            print(f"    Cell {cid}: {f:5,} / {len(grp):,}  "
                  f"({100*f/len(grp):.1f}%)")
        print(f"\n  Risk breakdown:")
        for lv in _RISK_LABELS:
            cnt = (results["risk_level"] == lv).sum()
            print(f"    {lv:<10s}: {cnt:7,}  ({100*cnt/total:.1f}%)")
        print(f"{'─'*55}\n")


# ══════════════════════════════════════════════════════════════════════════════
# CLI ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def _parse_args():
    p = argparse.ArgumentParser(
        description="RAN Cell Failure — Production Inference")
    p.add_argument("--data",   default=DEFAULT_DATA,
                   help="Input .parquet file (wide format)")
    p.add_argument("--model",  default=DEFAULT_MODEL,
                   help=".txt or .pkl model file")
    p.add_argument("--meta",   default=DEFAULT_META,
                   help="Feature manifest JSON from training")
    p.add_argument("--output", default=DEFAULT_OUTPUT,
                   help="Output CSV path")
    p.add_argument("--threshold", type=float, default=None,
                   help="Decision threshold (overrides meta JSON)")
    # FIX: use parse_known_args so Jupyter's kernel flags (e.g. -f kernel.json)
    # are silently ignored instead of raising SystemExit(2).
    args, _ = p.parse_known_args()
    return args


def main():
    args  = _parse_args()
    pred  = RANCellPredictor(args.model, args.meta, args.threshold)
    results = pred.predict(args.data, args.output)
    print(results.head(20).to_string(index=False))


# ── Colab / notebook convenience: run directly without CLI ───────────────────
if __name__ == "__main__":
    if _is_notebook():
        # Running inside Jupyter / Colab — skip argparse entirely
        print("Running with default paths (Colab/notebook mode) …\n")
        predictor = RANCellPredictor(DEFAULT_MODEL, DEFAULT_META)
        results   = predictor.predict(DEFAULT_DATA, DEFAULT_OUTPUT)
        print(results.head(30).to_string(index=False))
    else:
        main()

Running with default paths (Colab/notebook mode) …

[MODEL] Loaded: ran_cell_model.txt  (284 features  |  threshold=0.5)
[LOAD]  Detected format: Parquet
[LOAD]  10,000 site-rows × 188 columns
[TRANSFORM] Wide→Long: 30,000 cell-rows
[FEATURES] Encoding → temporal → rolling → lag → delta …
[ALIGN]  Feature matrix: (30000, 284)
[SCORE]  Scored 30,000 rows
[OUTPUT] Saved 30,000 rows → ran_predictions.csv

───────────────────────────────────────────────────────
  Total cell-hours scored : 30,000
  Predicted failures      : 40  (0.1%)

  By cell:
    Cell 1:     1 / 10,000  (0.0%)
    Cell 2:    14 / 10,000  (0.1%)
    Cell 3:    25 / 10,000  (0.2%)

  Risk breakdown:
    LOW       :  29,361  (97.9%)
    MEDIUM    :     630  (2.1%)
    HIGH      :       9  (0.0%)
    CRITICAL  :       0  (0.0%)
───────────────────────────────────────────────────────

                timestamp  site_id  cell_id  failure_probability  predicted_failure risk_level
2024-06-01 00:00:00+00:00 SITE_001        1    

In [14]:
import pandas as pd

# Read parquet file
df = pd.read_parquet("/content/my_test.parquet")

# Show as CSV (print to console)
print(df.to_csv(index=False))

# Or save to CSV file
df.to_csv("output.csv", index=False)

message_id,timestamp,sequence_number,scenario_label,site_id,site_name,latitude,longitude,region,vendor,bbu_status,bbu_op_state,bbu_cpu_utilization_percent,bbu_memory_utilization_percent,bbu_disk_usage_percent,bbu_process_latency_ms,bbu_active_users,bbu_control_plane_latency_ms,bbu_user_plane_latency_ms,ru_1_status,ru_1_op_state,ru_1_temperature_c,ru_1_tx_power_watts,ru_1_rx_signal_strength_dbm,ru_1_vswr,ru_1_current_ampere,ru_1_voltage_volt,ru_1_packet_error_rate,ru_1_throughput_mbps,ru_2_status,ru_2_op_state,ru_2_temperature_c,ru_2_tx_power_watts,ru_2_rx_signal_strength_dbm,ru_2_vswr,ru_2_current_ampere,ru_2_voltage_volt,ru_2_packet_error_rate,ru_2_throughput_mbps,ru_3_status,ru_3_op_state,ru_3_temperature_c,ru_3_tx_power_watts,ru_3_rx_signal_strength_dbm,ru_3_vswr,ru_3_current_ampere,ru_3_voltage_volt,ru_3_packet_error_rate,ru_3_throughput_mbps,ant_1_tilt_degree,ant_1_azimuth_degree,ant_1_mimo_layers,ant_1_status,ant_1_op_state,ant_1_rssi_dbm,ant_1_snr_db,ant_2_tilt_degree,ant_2_azim

8294400
